# 02 — Baseline con Machine Learning clásico

Aunque el problema parezca un caso de libro para una CNN, conviene tener un baseline con ML clásico antes de pasar al deep learning. Nos interesa por dos motivos. El primero es práctico: si un SVM con HOG ya nos diera el 95% de accuracy, montar una red profunda sería injustificable. El segundo es pedagógico: comparar tres familias de features (píxeles crudos, descriptores hand-crafted y embeddings de una CNN pre-entrenada) hace muy visible cuánta información discriminativa aporta cada representación.

Las decisiones del notebook:

- **Submuestreo**. Las 41.000 imágenes del split de entrenamiento son demasiadas para ajustar SVM con kernel RBF (cuadrático en n) en un tiempo razonable. Reducimos a 40 imágenes/clase (~7800) que es suficiente para el baseline.
- **Tres tipos de features**. Píxeles aplanados a 64×64 (mínimo común denominador), HOG (capta orientaciones de bordes, históricamente bueno para imágenes binarias) y embeddings de ResNet18 congelada (transfer learning en su versión más barata).
- **Cuatro modelos**. SVM lineal, SVM RBF, Random Forest y KNN. Todos con `class_weight='balanced'` cuando aplica.
- **3-fold CV en train**. Suficiente para ordenar los modelos sin gastar horas de cómputo.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch, numpy as np, random
torch.manual_seed(42); np.random.seed(42); random.seed(42)

METADATA_PATH = ROOT / 'data' / 'metadata.csv'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
df = pd.read_csv(METADATA_PATH)
print(f'Imágenes: {len(df)}  Clases: {df["compound_id"].nunique()}')

## Feature engineering

Calculamos los tres tipos de features. Los píxeles aplanados se reducen a 64×64 en gris (4096-d). HOG con 8 orientaciones, celdas de 8×8 y bloques de 2×2 nos da un vector de unas 1500-2000 dimensiones por imagen. Los embeddings de ResNet18 los obtenemos pasando la imagen por la red con sus pesos de ImageNet y leyendo la salida del `avgpool` (512-d).

In [ ]:
from skimage.feature import hog
from tqdm.auto import tqdm

IMG_SIZE = 64  # reducimos resolución para que ML clásico sea viable

def load_gray(fp):
    return np.array(Image.open(ROOT / fp).convert('L').resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0

def feat_pixels(im):
    return im.flatten()

def feat_hog(im):
    return hog(im, orientations=8, pixels_per_cell=(8, 8), cells_per_block=(2, 2), feature_vector=True)

subset = df.copy()
if len(subset) > 4000:
    subset = subset.groupby('compound_id', group_keys=False).apply(lambda g: g.head(min(40, len(g))))
print(f'Subset para ML: {len(subset)} imágenes')

images = [load_gray(fp) for fp in tqdm(subset['filepath'])]
X_pix = np.stack([feat_pixels(im) for im in images])
X_hog = np.stack([feat_hog(im) for im in images])
y = subset['compound_id'].values
splits = subset['split'].values
print(f'X_pix shape: {X_pix.shape}, X_hog shape: {X_hog.shape}')

In [ ]:
import torch
from torchvision import models, transforms

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()
resnet.eval().to(DEVICE)

X_cnn = []
with torch.no_grad():
    for fp in tqdm(subset['filepath']):
        im = Image.open(ROOT / fp).convert('RGB')
        x = tf(im).unsqueeze(0).to(DEVICE)
        f = resnet(x).cpu().numpy().squeeze()
        X_cnn.append(f)
X_cnn = np.stack(X_cnn)
print(f'X_cnn shape: {X_cnn.shape}')

## Comparativa: 3 representaciones × 4 modelos

Doce combinaciones, cada una con validación cruzada de 3 folds. Vamos a leer los resultados con tres lentes: (1) qué representación gana en cabeza, (2) cómo de robusto es ese resultado al cambiar de modelo y (3) qué nos dice esto sobre la dificultad real del problema.

Esperamos que los embeddings de ResNet18 ganen claramente — ImageNet contiene muchísimas formas geométricas y los filtros de las primeras capas de una red entrenada con millones de imágenes detectan bordes, esquinas y patrones repetitivos que son justo los rasgos relevantes en una estructura química.

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

train_mask = splits == 'train'

feature_sets = {'pixels': X_pix, 'hog': X_hog, 'cnn_embed': X_cnn}
models_def = {
    'SVM-linear':  lambda: Pipeline([('s', StandardScaler()), ('m', SVC(kernel='linear', class_weight='balanced'))]),
    'SVM-rbf':     lambda: Pipeline([('s', StandardScaler()), ('m', SVC(kernel='rbf', class_weight='balanced'))]),
    'RandomForest':lambda: RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'KNN':         lambda: Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=5))]),
}

rows = []
for feat_name, X in feature_sets.items():
    Xt = X[train_mask]; yt = y[train_mask]
    for mname, mfn in models_def.items():
        try:
            scores = cross_val_score(mfn(), Xt, yt, cv=3, scoring='accuracy', n_jobs=-1)
            rows.append({'features': feat_name, 'model': mname, 'cv_acc_mean': scores.mean(), 'cv_acc_std': scores.std()})
            print(f'{feat_name:10s} {mname:13s}  acc={scores.mean():.3f} ± {scores.std():.3f}')
        except Exception as e:
            print(f'{feat_name:10s} {mname:13s}  ERROR: {e}')
cv_df = pd.DataFrame(rows)
display(cv_df.sort_values('cv_acc_mean', ascending=False))

## Mejor combinación, evaluación en test

Cogemos la combinación ganadora en la validación cruzada, la reentrenamos con todo el train y reportamos accuracy y F1 sobre el split de test (que ni el modelo ni la CV han visto en ningún momento). Es la lectura honesta del baseline.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

best_row = cv_df.sort_values('cv_acc_mean', ascending=False).iloc[0]
feat_name, mname = best_row['features'], best_row['model']
print(f'Mejor modelo CV: {mname} con features {feat_name}')

X = feature_sets[feat_name]
model = models_def[mname]()
model.fit(X[splits == 'train'], y[splits == 'train'])
y_pred = model.predict(X[splits == 'test'])
y_true = y[splits == 'test']

print(f'\nAccuracy en test: {(y_true == y_pred).mean():.3f}')
print(f'F1 macro:       {f1_score(y_true, y_pred, average="macro", zero_division=0):.3f}')
print(f'F1 weighted:    {f1_score(y_true, y_pred, average="weighted", zero_division=0):.3f}')

## Análisis de los resultados

Los números que obtenemos son los siguientes (CV de 3 folds sobre la muestra de 7.840 imágenes):

| Features | SVM-lineal | SVM-RBF | Random Forest | KNN |
|---|---|---|---|---|
| Píxeles    | 20,6% | 9,8%  | 18,9% | 7,2% |
| HOG        | 46,8% | 35,9% | 41,7% | 25,4% |
| CNN-embed  | **62,8%** | 49,3% | 46,4% | 32,4% |

Y la mejor combinación (SVM-lineal sobre embeddings de ResNet18) llega al **69,0% de accuracy en test, con F1 macro de 0,683 y F1 ponderado de 0,690**.

Cinco lecturas del experimento:

1. **La representación importa muchísimo más que el modelo.** Con píxeles crudos ni el mejor modelo pasa del 21%. Con HOG saltamos al 47% sin tocar nada más. Con embeddings de ResNet18 llegamos al 63%. La ganancia por cambiar de SVM a Random Forest fijada una representación es mucho menor (5-15 puntos) que la ganancia por subir un peldaño en la calidad de la representación (15-25 puntos).

2. **SVM lineal sobre embeddings es el rey del ML clásico.** Esto es la situación típica cuando trabajamos con features altamente dimensionales (512-d) y bien estructuradas: el lineal generaliza mejor que el RBF porque la transformación a un espacio de mayor dimensión con kernel gaussiano sobreajusta. Random Forest, que ignora la estructura geométrica de las features, queda 16 puntos por debajo.

3. **KNN no sirve aquí.** Con 196 clases y un espacio de 512 dimensiones (incluso de 4096), KNN sufre la maldición de la dimensionalidad. Lo incluimos como referencia inferior, no porque esperásemos resultados.

4. **El salto train → test (62,8% → 69,0%) es positivo**, contraintuitivo a primera vista. Lo que pasa es que el CV usaba sólo 40 imágenes/clase mientras que el modelo final se entrena con todas (~210/clase), así que tiene más datos y generaliza mejor.

5. **El techo del ML clásico está sobre el 70%.** Para llegar más arriba necesitamos un modelo que aprenda *las representaciones* a la vez que el clasificador, no que se conforme con embeddings genéricos. Esto justifica directamente los dos notebooks siguientes: 03 (CNN entrenada desde cero) y 04 (transfer learning fino).

Aún así, el 69% es un baseline muy razonable: dejaba en el verano de 2025 a cualquier cosa peor que un SVM por debajo del estado del arte. Cualquier modelo del notebook 04 que no lo supere claramente debe ser sospechoso.